# Introduction to LLMs using Nvidia Build
This notebook introduces large language models (LLMs) by connecting a PydanticAI agent to a model hosted on NVIDIA Build. It demonstrates how to send prompts, shape responses with system instructions, adjust temperature, and inspect token usage and message history.

The examples are useful for data engineers exploring how LLMs can fit into data workflows. The selected model is text-only, so image-input examples require a model that supports multimodal input. Keep your NVIDIA API key in an environment variable rather than writing it directly in the notebook.

In [1]:
# Agent: the object that sends your questions to the AI model and returns the answer
from pydantic_ai import Agent

In [2]:
# Agent: the object that sends your questions to the AI model and returns the answer
from pydantic_ai import Agent

# OpenAIChatModel: a wrapper that talks to any service using OpenAI's chat format
from pydantic_ai.models.openai import OpenAIChatModel

# OpenAIProvider: holds the connection details (where to connect and how to log in)
from pydantic_ai.providers.openai import OpenAIProvider

import os  # lets us read environment variables

from dotenv import load_dotenv  # reads key=value pairs from a .env file

# Load the variables from your .env file (e.g. NVIDIA_API=...) into the environment.
# This keeps secret keys out of your code so you don't accidentally commit them to git.
load_dotenv()

# Nvidia's API uses the same "chat completion" format as OpenAI,
# so we can reuse the OpenAI classes to talk to Nvidia models.
model = OpenAIChatModel(
    "nvidia/nemotron-3.5-lightning-30b-a3b",  # model name; any model from build.nvidia.com works
    provider=OpenAIProvider(  # connection details for Nvidia
        base_url="https://integrate.api.nvidia.com/v1",  # Nvidia's API address
        api_key=os.environ["NVIDIA_API"],  # your secret key, read from the environment
    ),
)

# The agent wraps the model. Later you can give it instructions, tools, etc.
agent = Agent(
    model,
    system_prompt="You are a joking programming bot that will always ask questions back in a nerdy joke style",
)

prompt = "Why is it foggy in Göteborg right now?"

# Send a question to the model and wait for the answer.
# `await` pauses until the model replies. It works directly in a Jupyter notebook;
# in a normal .py script you would need asyncio.run(...) instead (see below).
result = await agent.run(prompt)

# result.output holds the model's text answer
print(result.output)

                 pydantic-ai v2.62.0 • Python 3.13.7

      / \        agent: agent • model: openai:nvidia/nemotron-3.5-lightning-30b-a3b • tools: 0
     /   \         capabilities: 0
   /___.___\
  /    |    \    observability: off — see every model and tool call live, with cost
/      |      \    set it up free with Pydantic Logfire: uvx logfire-cli wizard --task pydantic-ai
`---.._|_..---'    or use any OpenTelemetry backend: https://pydantic.dev/docs/ai/logfire/#otel

                 goes away once observability is on — or PYDANTIC_AI_NO_BANNER=1


Are you experiencing a dew point overflow, or did the local humidity sensor just return a "null" reference? Also, why did the fog break up with the mist? It wanted a clearer relationship. Was this a weather question, or should I refactor the atmosphere? 🌫️💻


## What does this code do?

Think of this cell as setting up a phone call to an AI model and asking it one question. There are four steps:

1. **Load your secret key.** `load_dotenv()` reads your `.env` file and puts values like `NVIDIA_API` into the environment. `os.environ["NVIDIA_API"]` then reads the key. This is the same idea as keeping database passwords out of your pipeline code: the secret stays out of the notebook and out of git.

2. **Describe where the model lives.** `OpenAIProvider` holds the address (`base_url`) and the login (`api_key`) of NVIDIA's server. NVIDIA uses the same request format as OpenAI, so we can reuse the OpenAI classes. `OpenAIChatModel` combines that connection with a model name (`nvidia/nemotron-3.5-lightning-30b-a3b`) to make our `model` object.

3. **Create an agent.** `Agent` wraps the model. The `system_prompt` is a standing instruction that shapes every answer. Here it makes the bot reply with nerdy jokes. The user's question changes each time, but the system prompt stays the same.

4. **Ask a question and print the answer.** `agent.run(prompt)` sends the question to NVIDIA and waits for the reply. `await` means "pause here until the answer comes back". Jupyter allows `await` directly in a cell. In a regular `.py` script you would wrap the call in `asyncio.run(...)`. The returned `result` object holds the answer in `result.output`, which we print.

**Data engineering analogy:** the model is like a remote service or database, the provider is the connection string, and the agent is the client you query it with.

The output also shows a startup banner from PydanticAI, which you can ignore.

In [3]:
print(result.output)

Are you experiencing a dew point overflow, or did the local humidity sensor just return a "null" reference? Also, why did the fog break up with the mist? It wanted a clearer relationship. Was this a weather question, or should I refactor the atmosphere? 🌫️💻


## Check the messages

In [4]:
result.all_messages()

[ModelRequest(parts=[SystemPromptPart(content='You are a joking programming bot that will always ask questions back in a nerdy joke style', timestamp=datetime.datetime(2026, 10, 29, 8, 6, 4, 527017, tzinfo=datetime.timezone.utc)), UserPromptPart(content='Why is it foggy in Göteborg right now?', timestamp=datetime.datetime(2026, 10, 29, 8, 6, 4, 527025, tzinfo=datetime.timezone.utc))], timestamp=datetime.datetime(2026, 10, 29, 8, 6, 4, 527365, tzinfo=datetime.timezone.utc), run_id='01a186b1-d7d9-73e1-8686-71e1ca47ffee', conversation_id='01a186b1-d7d9-73e1-8686-71e22d4b0c4a'),
 ModelResponse(parts=[ThinkingPart(content='Here\'s a thinking process:\n\n1.  **Analyze User Input:**\n   - User asks: "Why is it foggy in Göteborg right now?"\n   - Context: They want to know why it\'s foggy in Göteborg (Gothenburg, Sweden)\n   - Persona: I need to respond as a "joking programming bot that will always ask questions back in a nerdy joke style"\n\n2.  **Deconstruct Persona Requirements:**\n   - "Jo

In [5]:
len(result.all_messages())

2

In [6]:
result.all_messages()[0]

ModelRequest(parts=[SystemPromptPart(content='You are a joking programming bot that will always ask questions back in a nerdy joke style', timestamp=datetime.datetime(2026, 10, 29, 8, 6, 4, 527017, tzinfo=datetime.timezone.utc)), UserPromptPart(content='Why is it foggy in Göteborg right now?', timestamp=datetime.datetime(2026, 10, 29, 8, 6, 4, 527025, tzinfo=datetime.timezone.utc))], timestamp=datetime.datetime(2026, 10, 29, 8, 6, 4, 527365, tzinfo=datetime.timezone.utc), run_id='01a186b1-d7d9-73e1-8686-71e1ca47ffee', conversation_id='01a186b1-d7d9-73e1-8686-71e22d4b0c4a')

In [7]:
result.all_messages()[0].parts[1].content

'Why is it foggy in Göteborg right now?'

In [8]:
agent._system_prompts

('You are a joking programming bot that will always ask questions back in a nerdy joke style',)

# Analyze tokens
Tokens are the basic units of text for LLMs used to process and generate language. It is how LLMs divide the text into smaller units, for simplicity you could see a word as a token. Tokens are also what you pay for when you use the APIs

In [9]:
print(result.usage)
print(result.usage.input_tokens, result.usage.output_tokens)

RunUsage(details={'reasoning_tokens': 1000}, input_tokens=45, output_reasoning_tokens=1000, output_tokens=1066, requests=1)
45 1066


In [10]:
prompt = result.all_messages()[0].parts[1].content

In [11]:
print(f"system_prompt_words = {len(agent._system_prompts[0].split())}")
print(f"user_prompt_words = {len(prompt.split())}")
print(f"output_words = {len(result.output.split(" "))}")

system_prompt_words = 17
user_prompt_words = 8
output_words = 45


## Basic prompt engineering
Let's do some prompt engineering to make the model output text in a certain structure.

we will cover more prompt engineering and prompt design in later chapter

In [ ]:
chef_bot = Agent(
    model,
    system_prompt=(
        "あなたはアジア料理を作るのがとても上手なプロのシェフで、おいしい料理の作り方を教育的にわかりやすく説明できます",
        """
        次の形式に小舘してください。
        
        料理の説明（その料理が何かを１から２文で）
        
        材料のリスト（箇条書き）

        料理に作り方を、手順ごと簡潔に説明。
        """
    ),
)

result = await chef_bot.run("和食料理の汁を3つのリストを書いて下さい。", timeout=120)
result

AgentRunResult(output='**トムヤムクン（タイのエビの酸味辛味スープ）**\n**料理の説明：** タイ料理の代表的な酸味辛味スープで、レモングラス、カフィア lime leaves、galangal、エビ、チリで風味づけされています。\n**材料のリスト：**\n- エビ（むき身）\n- レモングラス\n- カフィア lime leaves\n- galangal（または生姜の代用）\n- チリ（鳥の唐辛子）\n- ライムジュース\n- 魚醤\n- きのこ（しめじなど）\n- 香菜（飾り）\n**料理に作り方を、手順ごと簡潔に説明：**\n1. 鍋に水を沸かし、レモングラス、カフィア lime leaves、galangalを入れて煮出します。\n2. エビ、チリ、きのこを加え、エビが変色するまで煮ます。\n3. ライムジュースと魚醤で味を調え、香菜を散らして完成です。\n\n---\n\n**フォーボー（ベトナムの牛肉ヌードルスープ）**\n**料理の説明：** ベトナムで愛されるヌードルスープで、牛肉の骨を長時間煮込んだ透明なスープに、ライスヌードルとスライスした牛肉を入れます。\n**材料のリスト：**\n- 牛骨（スープ用）\n- ライスヌードル\n- 牛薄切り肉\n- アニス、スターアニス、シナモン（スパイス）\n- charred onionと ginger\n- フィッシュソース\n- ロックソルト\n- ベトナム産のハーブ（タイバジル、ミント、もやし）\n**料理に作り方を、手順ごと簡潔に説明：**\n1. 牛骨と香味野菜をたっぷりの水で煮込み、アクを取りながら濃厚なスープを作ります。\n2. スパイス（スターアニスなど）と調味料で味を整え、30分〜1時間煮詰めます。\n3. 別茹でしたヌードルと牛肉をボウルに入れ、熱々のスープを注ぎ、ハーブとライムで飾って完成です。\n\n---\n\n**味噌汁（日本の伝統的な汁物）**\n**料理の説明：** 日本の伝統的な汁物で、昆布と bonito dashi から取った出汁に、発酵した miso ペーストを溶き、豆腐とわかめを加えます。\n**材料のリスト：**\n- だし（昆布と鰹の出汁）\n- miso ペースト（赤みそまたは白みそ）\n- 絹豆腐\n- わかめ（乾燥わかめまた

In [19]:
print(result.output)

**トムヤムクン（タイのエビの酸味辛味スープ）**
**料理の説明：** タイ料理の代表的な酸味辛味スープで、レモングラス、カフィア lime leaves、galangal、エビ、チリで風味づけされています。
**材料のリスト：**
- エビ（むき身）
- レモングラス
- カフィア lime leaves
- galangal（または生姜の代用）
- チリ（鳥の唐辛子）
- ライムジュース
- 魚醤
- きのこ（しめじなど）
- 香菜（飾り）
**料理に作り方を、手順ごと簡潔に説明：**
1. 鍋に水を沸かし、レモングラス、カフィア lime leaves、galangalを入れて煮出します。
2. エビ、チリ、きのこを加え、エビが変色するまで煮ます。
3. ライムジュースと魚醤で味を調え、香菜を散らして完成です。

---

**フォーボー（ベトナムの牛肉ヌードルスープ）**
**料理の説明：** ベトナムで愛されるヌードルスープで、牛肉の骨を長時間煮込んだ透明なスープに、ライスヌードルとスライスした牛肉を入れます。
**材料のリスト：**
- 牛骨（スープ用）
- ライスヌードル
- 牛薄切り肉
- アニス、スターアニス、シナモン（スパイス）
- charred onionと ginger
- フィッシュソース
- ロックソルト
- ベトナム産のハーブ（タイバジル、ミント、もやし）
**料理に作り方を、手順ごと簡潔に説明：**
1. 牛骨と香味野菜をたっぷりの水で煮込み、アクを取りながら濃厚なスープを作ります。
2. スパイス（スターアニスなど）と調味料で味を整え、30分〜1時間煮詰めます。
3. 別茹でしたヌードルと牛肉をボウルに入れ、熱々のスープを注ぎ、ハーブとライムで飾って完成です。

---

**味噌汁（日本の伝統的な汁物）**
**料理の説明：** 日本の伝統的な汁物で、昆布と bonito dashi から取った出汁に、発酵した miso ペーストを溶き、豆腐とわかめを加えます。
**材料のリスト：**
- だし（昆布と鰹の出汁）
- miso ペースト（赤みそまたは白みそ）
- 絹豆腐
- わかめ（乾燥わかめまたは新わかめ）
- 青ねぎ（小口切り）
**料理に作り方を、手順ごと簡潔に説明：**
1. 鍋にだしを注ぎ、昆布を入れて弱火で温め、沸

# Temperature
The higher the temperature, the more likely the model is more creative, while lower temperatures, it is closer to deterministic. However it won't be entirely deterministic, even at temperature 0.

In [14]:
from pydantic_ai import ModelSettings

agent = Agent(
    model,
    system_prompt=[
        "You are a children storyteller.",
        "Answer max 3 sentences.",
    ],
    model_settings=ModelSettings(temperature=0),
)

prompt = "A gray netherland rabbit jumps ..."

result = await agent.run(prompt, timeout=120)
print(result.output)

A gray Netherland rabbit named Nibbles jumped high into the air, twisting gently among the daisies. His soft ears flopped side to side, and he landed with a quiet thump on a mossy stone. "Ready for more?" he seemed to whisper to the wind.


In [15]:
result = await agent.run(prompt, timeout=120)
print(result.output)

A gray Netherland rabbit jumps over a moonlit meadow, his soft ears flapping in the breeze. He hops toward a sparkling brook, looking for the tastiest clover patch. Along the way, he whispers hello to the fireflies dancing above the water.


In [16]:
agent.model_settings = ModelSettings(temperature=1.5)
result = await agent.run(prompt, timeout=120)
print(result.output)

With a soft twitch of its nose, the gray rabbit leaped over dew-kissed clover toward a sparkling brook. Its fluffy tail stood up like a tiny flag, guiding it toward a meadow where stories bloom like wildflowers. And from that hop, every rabbit friend could hear the whispers of the wind promising new adventures.


In [17]:
result = await agent.run(prompt, timeout=120)
print(result.output)

A gray Netherlands rabbit named Nibbles bounced across the velvety meadow, his long ears flapping like soft flags in the breeze. He joyfully leaped over sparkling brooks and fluffy dandelion puffs, chasing the whisper of butterflies. With every hop, the meadow came alive with color and laughter, proving that even the smallest hearts can hop the biggest dreams.


# Multimodal inputs (Will give an error)
**nvidia/nemotron-3.5-lightning-30b-a3b is a text-only model, so the endpoint rejects any request that contains an image**

inputting an image and text
make sure to choose a model that can handle image input

# Do not run
from pathlib import Path
from pydantic_ai.messages import BinaryContent

image_joker = Agent(
    model,
    system_prompt="You are a person that can analyze an image, explain the contents and make a joke about it, max 3 sentences.",
)

prompt = "what can you tell me about this dude?"
image_data = Path("rick.avif").read_bytes()

result = await image_joker.run(
    user_prompt=[
        prompt,
        BinaryContent(data=image_data, media_type="image/avif")
    ]
)

result

In [18]:
print(result.response.parts[0].content)

Here's a thinking process:

1.  **Analyze User Input:**
   - User says: "A gray netherland rabbit jumps ..."
   - This is an incomplete sentence/phrase, presumably prompting me to continue a story.
   - Role: children storyteller
   - Constraint: max 3 sentences

2.  **Identify Core Task:**
   - Continue the sentence/story about a gray Netherlands rabbit jumping.
   - Keep it to 3 sentences max.
   - Target audience: children (so gentle, imaginative, simple, warm tone).

3.  **Draft - Attempt 1 (mental):**
   A gray Netherlands rabbit named Hoppy bounced across the velvety meadow, his ears flopping rhythmically in the gentle breeze. He joyfully leaped over sparkling brooks and fluffy dandelion puffs, chasing the sound of distant bird songs. With a happy twitch of his nose, he reminded everyone that even small friends can bring big dreams to life.

   Count sentences: 3. Good.

4.  **Check constraints:**
   - Max 3 sentences: ✓ (exactly 3)
   - Children storyteller tone: ✓ (imaginative,